# Phase 2: Data Cleaning & Preparation
Predictive Maintenance of Industrial Equipment using IoT Sensor Data

**Goal:** handle missing values, duplicates, invalid values, and outliers; convert data types correctly; produce a clean analytical dataset for EDA and modeling.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw_sensor_data.csv')
print(df.shape)
df.head()

(86430, 12)


,Machine_ID,Timestamp,Temperature,Pressure,Vibration,RPM,Current,Voltage,Operating_Hours,Maintenance_Count,Machine_Age,Failure
0,M105,2026-07-13 15:00:00,68.28,135.96,0.386,2838.7,14.11,429.8,4185,13,2,0
1,M121,2026-07-24 02:00:00,63.36,142.70,0.363,2571.8,10.06,412.1,4665,4,6,0
2,M109,2026-05-21 04:00:00,56.52,135.59,0.360,2850.0,12.43,404.8,2325,12,3,0
3,M109,2026-05-05 09:00:00,57.10,133.10,0.344,2815.4,12.28,406.8,1946,12,3,0
4,M101,2026-07-16 17:00:00,59.85,152.18,0.341,2770.8,10.01,397.8,3389,9,5,0


## 2.1 Inspect dataset size, columns, data types

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 86430 entries, 0 to 86429
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Machine_ID         86430 non-null  str    
 1   Timestamp          86430 non-null  str    
 2   Temperature        86085 non-null  float64
 3   Pressure           86085 non-null  float64
 4   Vibration          86085 non-null  float64
 5   RPM                86430 non-null  float64
 6   Current            86430 non-null  float64
 7   Voltage            86085 non-null  float64
 8   Operating_Hours    86430 non-null  int64  
 9   Maintenance_Count  86430 non-null  int64  
 10  Machine_Age        86430 non-null  int64  
 11  Failure            86430 non-null  int64  
dtypes: float64(6), int64(4), str(2)
memory usage: 7.9 MB


## 2.2 Convert Timestamp to datetime

In [3]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.sort_values(['Machine_ID','Timestamp']).reset_index(drop=True)
df.dtypes

Machine_ID                      str
Timestamp            datetime64[us]
Temperature                 float64
Pressure                    float64
Vibration                   float64
RPM                         float64
Current                     float64
Voltage                     float64
Operating_Hours               int64
Maintenance_Count             int64
Machine_Age                   int64
Failure                       int64
dtype: object

## 2.3 Check missing values

In [4]:
df.isnull().sum()

Machine_ID             0
Timestamp              0
Temperature          345
Pressure             345
Vibration            345
RPM                    0
Current                0
Voltage              345
Operating_Hours        0
Maintenance_Count      0
Machine_Age            0
Failure                0
dtype: int64

**Decision:** Temperature, Pressure, Vibration, and Voltage have a small number of missing readings (sensor dropout, common in IoT systems). Since each machine has continuous hourly readings, we fill missing sensor values using **forward-fill within each machine's own timeline** (carry the last known reading forward) rather than a global mean, which would ignore each machine's individual operating range.

In [5]:
sensor_cols = ['Temperature','Pressure','Vibration','RPM','Current','Voltage']
df[sensor_cols] = df.groupby('Machine_ID')[sensor_cols].transform(lambda x: x.ffill().bfill())
df.isnull().sum()

Machine_ID           0
Timestamp            0
Temperature          0
Pressure             0
Vibration            0
RPM                  0
Current              0
Voltage              0
Operating_Hours      0
Maintenance_Count    0
Machine_Age          0
Failure              0
dtype: int64

## 2.4 Check and remove duplicate records

In [6]:
print('Duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print('Shape after removing duplicates:', df.shape)

Duplicate rows: 30
Shape after removing duplicates: (86400, 12)


## 2.5 Detect and handle outliers / invalid sensor values
- Vibration cannot physically be negative -> invalid readings, take absolute value.
- Temperature has a few extreme spikes (sensor glitches) -> cap using the IQR method rather than deleting rows, since deleting would remove real machine context.

In [7]:
# Fix invalid negative vibration
invalid_vib = (df['Vibration'] < 0).sum()
print('Invalid negative vibration readings:', invalid_vib)
df['Vibration'] = df['Vibration'].abs()

# Cap Temperature outliers using IQR method
Q1, Q3 = df['Temperature'].quantile([0.25, 0.75])
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
outliers = (df['Temperature'] > upper_bound).sum()
print('Temperature outliers capped:', outliers, '| upper bound:', round(upper_bound,2))
df['Temperature'] = np.where(df['Temperature'] > upper_bound, upper_bound, df['Temperature'])

Invalid negative vibration readings: 15
Temperature outliers capped: 1302 | upper bound: 76.38


## 2.6 Final sanity check

In [8]:
df.describe().T[['min','max','mean']]

,min,max,mean
Timestamp,2026-05-01 00:00:00,2026-07-29 23:00:00,2026-06-14 23:30:00
Temperature,50.69,76.38,61.483989
Pressure,90.39,167.67,139.714479
Vibration,0.06,2.661,0.401397
RPM,2159.1,3177.0,2825.582453
Current,7.01,22.37,11.513542
Voltage,373.9,436.1,415.511601
Operating_Hours,678.0,5139.0,2997.125
Maintenance_Count,0.0,14.0,6.825
Machine_Age,1.0,11.0,6.55


In [9]:
print('Failure distribution:')
print(df['Failure'].value_counts(normalize=True))

Failure distribution:
Failure
0    0.991424
1    0.008576
Name: proportion, dtype: float64


## 2.7 Save the cleaned dataset

In [10]:
df.to_csv('../data/cleaned_sensor_data.csv', index=False)
print('Saved cleaned_sensor_data.csv, shape:', df.shape)

Saved cleaned_sensor_data.csv, shape: (86400, 12)
